# 00 · Preparación del entorno

**Tema 18 · IA Generativa Avanzada** · EBIS Business Techschool

Ejecuta este notebook **antes de la primera clase** (25/11). Tarda unos 2 minutos y comprueba que:

1. Las librerías del curso están instaladas.
2. Tu API key de OpenAI está bien configurada (sin mostrarla en pantalla).
3. Puedes llamar al modelo `gpt-6-luna` y ver cuánto cuesta una llamada.

Si al final ves **✅ Todo listo**, ya estás. Si algo falla, el mensaje de error te dice qué revisar.

## 1 · Instalar las librerías

En Colab tarda unos segundos. En local, mejor dentro de un entorno virtual.

In [ ]:
!pip install -q --upgrade openai tiktoken pandas

In [ ]:
import sys, importlib.metadata as md

print("Python", sys.version.split()[0])
for paquete in ["openai", "tiktoken", "pandas"]:
    print(f"{paquete:10s}", md.version(paquete))

assert sys.version_info >= (3, 10), "Necesitas Python 3.10 o superior (Colab ya lo cumple)"
assert int(md.version("openai").split(".")[0]) >= 2, "Actualiza openai: pip install --upgrade openai"

## 2 · Leer la API key sin escribirla en el notebook

El orden de búsqueda es:

1. **Secretos de Colab** (icono de la llave, a la izquierda) con el nombre `OPENAI_API_KEY`.
2. **Variable de entorno** `OPENAI_API_KEY` en tu ordenador.
3. Si no la encuentra, te la **pide** con `getpass` (no se ve al escribir y no se guarda en el notebook).

> ⚠️ Nunca pegues la clave directamente en una celda: si compartes o subes el notebook a GitHub, la estarás publicando.

In [ ]:
import os
from getpass import getpass

def cargar_api_key():
    try:  # 1) Secretos de Colab
        from google.colab import userdata
        clave = userdata.get("OPENAI_API_KEY")
        if clave:
            return clave, "secretos de Colab"
    except Exception:
        pass
    if os.environ.get("OPENAI_API_KEY"):  # 2) Variable de entorno
        return os.environ["OPENAI_API_KEY"], "variable de entorno"
    return getpass("Pega tu API key de OpenAI: "), "getpass"  # 3) Preguntar

clave, origen = cargar_api_key()
os.environ["OPENAI_API_KEY"] = clave.strip()
print(f"Clave leída desde {origen}: {clave[:7]}…{clave[-4:]}  ({len(clave)} caracteres)")
assert clave.startswith("sk-"), "La clave debería empezar por 'sk-'. ¿La has copiado entera?"

## 3 · Primera llamada al modelo

Usamos la **Responses API** con `gpt-6-luna` y un nivel de razonamiento bajo (`effort: low`), que es la configuración de todo el curso.

In [ ]:
import time
from openai import OpenAI

MODELO = "gpt-6-luna"
ESFUERZO = "low"
client = OpenAI()

t0 = time.time()
r = client.responses.create(
    model=MODELO,
    instructions="Eres el asistente del Tema 18 del máster de EBIS. Responde en español, en una frase.",
    input="Dame la bienvenida al tema de IA generativa avanzada.",
    reasoning={"effort": ESFUERZO},
)
print(r.output_text)
print(f"\n⏱️ {time.time() - t0:.1f} s")

## 4 · ¿Cuánto ha costado?

Cada respuesta trae el **uso de tokens**. Los tokens de razonamiento (lo que el modelo «piensa» antes de contestar) se cobran como salida aunque no los veas.

Precios de `gpt-6-luna` en noviembre de 2026, en dólares por millón de tokens (tarifa estándar). Si cambian, los tienes en openai.com/api/pricing.

In [ ]:
PRECIO_INPUT = 0.10    # $ por millón de tokens de entrada
PRECIO_OUTPUT = 0.50   # $ por millón de tokens de salida (incluye el razonamiento)

u = r.usage
razonamiento = u.output_tokens_details.reasoning_tokens
coste = u.input_tokens / 1e6 * PRECIO_INPUT + u.output_tokens / 1e6 * PRECIO_OUTPUT

print(f"Tokens de entrada : {u.input_tokens}")
print(f"Tokens de salida  : {u.output_tokens}  (de ellos, razonamiento: {razonamiento})")
print(f"Coste aproximado  : {coste:.6f} $  →  {1 / coste:,.0f} llamadas como esta por 1 $" if coste else "")

## 5 · Tokens: lo que de verdad lee el modelo

Los LLMs no leen palabras, leen **tokens**. `tiktoken` nos da una buena aproximación en local (en clase veremos cómo pedir el número exacto a la API). Fíjate en que el español suele gastar más tokens que el inglés para decir lo mismo.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")  # codificación de los modelos recientes de OpenAI
for frase in ["Los modelos de lenguaje no entienden palabras, entienden tokens.",
              "Language models don't understand words, they understand tokens."]:
    ids = enc.encode(frase)
    print(f"{len(ids):3d} tokens · {[enc.decode([i]) for i in ids]}")

## 6 · Resultado

In [ ]:
print("✅ Todo listo: librerías instaladas, API key configurada y modelo respondiendo.")
print("Nos vemos el miércoles 25/11 a las 18:30.")

---
**¿Algo ha fallado?**

| Error | Qué significa | Solución |
|---|---|---|
| `AuthenticationError` / 401 | La clave no es válida | Revisa que la has copiado entera, sin espacios |
| `RateLimitError` / 429 | Demasiadas peticiones o sin saldo | Espera un minuto; si sigue, avisa al profesor |
| `NotFoundError` / 404 con el modelo | El nombre del modelo no existe para esa clave | Comprueba `MODELO = "gpt-6-luna"` |
| `ModuleNotFoundError` | Falta una librería | Vuelve a ejecutar la celda de `pip install` |

Si no lo consigues, escribe al profesor con una captura del error **(sin que se vea tu clave)**.